In [ ]:
import os
import sys

if os.getcwd().endswith("notebooks"):
    os.chdir("..")

sys.path.append(os.path.abspath("./"))

print(f"Current work directory: {os.getcwd()}")

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata
import joblib
import scipy.sparse as sp
import h5py
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import scripts.plotting as pl
import scripts.utils as ut

In [ ]:
print("Loading GSM2406677 data...")
data_dir = r"./Data/GSM2406677"

# Load matrix, genes, barcodes
adata = sc.read_mtx(os.path.join(data_dir, "matrix.mtx.txt.gz")).T
genes = pd.read_csv(os.path.join(data_dir, "genes.tsv.gz"), header=None, sep='\t')
adata.var_names = genes[1].values
adata.var['gene_ids'] = genes[0].values
adata.var_names_make_unique()

barcodes = pd.read_csv(os.path.join(data_dir, "barcodes.tsv.gz"), header=None)
adata.obs_names = barcodes[0].values

# Extract gemgroup 1 (Tunicamycin)
gemgroup1_mask = adata.obs_names.str.endswith("-1")
adata = adata[gemgroup1_mask].copy()
print(f"Extracted {len(adata)} cells from gemgroup 1 (Tunicamycin).")

# Cast to object type
adata.obs_names = adata.obs_names.astype(object)
adata.var_names = adata.var_names.astype(object)

output_path = r"./Data/h5ad/GSM2406677_tunicamycin.h5ad"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
anndata.settings.allow_write_nullable_strings = True
adata.write_h5ad(output_path)

In [ ]:
print("Applying QC metrics and filtering...")
adata = ut.add_qc_metrics(adata)
adata = ut.basic_qc_filter(adata, pct_mt_max=20.0)
if 'oc43' in adata.var:
    adata = adata[:, ~adata.var['oc43']].copy()
adata = ut.normalize_log1p(adata)

genes_of_interest = ['TPI1','PPIA','HMGN2','FTL','RPS29','TSC22D3','SNHG7','HSPA8','ATP5J2','IFRD1']
valid_genes = [gene for gene in genes_of_interest if gene in adata.var_names]
adata_sub = adata[:, valid_genes]

df_input = adata_sub.to_df()
missing_genes = [gene for gene in genes_of_interest if gene not in df_input.columns]
for gene in missing_genes:
    df_input[gene] = 0.0

df_input.to_csv('./CSV/GSM2406677_tunicamycin_test.csv', index=False)